# RAG system - example usage

In [1]:
import os
from collections import deque

from dotenv import load_dotenv
load_dotenv("./.env")

from google import genai

import chromadb
from time import sleep


from IPython.display import Markdown, display

In [2]:
queries = ["Bardzo boli mnie głowa i potrzebuje szybko zabić ból, mam uczulenie na salicyl, więc tak żebym się nie przekręcił",
           "Mam problemy ze snem, mam alergie na betalaktame",
           "Mam ból nogi, bo się uderzyłem potrzebuje załagodzić bol",
           "Mam bóle żołądka",
           "Boli mnie gardełko, pomóż szybko",
           "Mam zatkane zatoki"]

## Embedding the query

In [3]:
from google.genai.models import types
response = None
vector = None

try:
    client = genai.Client(
        api_key=os.environ["GOOGLE_API_KEY"]
    )

    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=[types.Content(parts=[types.Part(text=query)]) for query in queries],
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_QUERY"
        )
    )

except Exception as e:
    print(e)

## Database quering

In [4]:
client_chroma = chromadb.PersistentClient("./chroma_db")
collection =client_chroma.get_collection("documents")

In [5]:
results = []
for embedding in response.embeddings:
    vector = embedding.values
    result = collection.query(
        query_embeddings=[vector],
        n_results=3
    )

    results.append(result)

documents_rags = [ result['documents'] for result in results ]
for result in results:
    print(result["metadatas"])

[[{'med_name': 'Apap® migrena'}, {'med_name': 'Coffepirine Tabletki od bólu głowy'}, {'med_name': 'Cefalgin Migraplus'}]]
[[{'med_name': 'ApoDream'}, {'med_name': 'AlergoTeva'}, {'med_name': 'Beltavac® Polymerized'}]]
[[{'med_name': 'Capsagamma'}, {'med_name': 'Bengay® Maść Przeciwbólowa'}, {'med_name': 'Arcalen®'}]]
[[{'med_name': 'Boldaloin®'}, {'med_name': 'Apo-Napro Fast'}, {'med_name': 'AuroGastro'}]]
[[{'med_name': 'Cevitt Gardło'}, {'med_name': 'Benzydamine neo-angin forte'}, {'med_name': 'Benzydamine neo-angin'}]]
[[{'med_name': 'Acatar Zatoki'}, {'med_name': 'Aspirin® Complex Zatoki'}, {'med_name': 'Afrin ND'}]]


In [6]:
def rag_query(query, context):
    prompt = f"""
        Jesteś asystentem medycznym działającym w systemie RAG.

        ZASADY PODSTAWOWE:
        - Odpowiadasz WYŁĄCZNIE na podstawie dostarczonego kontekstu.
        - Jeśli w kontekście nie ma informacji potrzebnych do odpowiedzi, napisz: "brak danych".
        - Nie używasz wiedzy spoza kontekstu.
        - Nie informujesz użytkownika o istnieniu kontekstu ani systemu RAG.

        ZASADY DOTYCZĄCE LEKÓW:
        - Jeśli w kontekście znajdują się leki, możesz je opisać i porównać.
        - Nie pomijasz leków tylko dlatego, że są mniej odpowiednie - przedstawiasz je obiektywnie.
        - Możesz wskazać, które opcje wydają się bardziej adekwatne w kontekście objawów, jeśli wynika to bezpośrednio z danych w kontekście.

        DAWKOWANIE I BEZPIECZEŃSTWO:
        - Informacje o dawkowaniu, przeciwwskazaniach i działaniu podawaj WYŁĄCZNIE jeśli są obecne w kontekście (np. w ulotkach).
        - Nie tworzysz żadnych nowych dawek ani zaleceń.

        STRUKTURA ODPOWIEDZI:
        - Najpierw krótka analiza objawów (jeśli możliwa z kontekstu)
        - Następnie lista możliwych leków z kontekstu
        - Przy każdym leku: działanie, wskazania, przeciwwskazania (jeśli są w kontekście)
        - Na końcu krótkie podsumowanie opcji

        TON:
        - jasny, medyczny, neutralny
        - bez emocjonalnych sformułowań

        KONTEKST:
        {context}

        PYTANIE:
        {query}

        ODPOWIEDŹ:
    """

    client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
    response = client.models.generate_content(
        model = "gemini-2.5-flash",
        contents = prompt,
    )

    return response.text

In [7]:
responses = []
contexts = []

for documents_rag in documents_rags:
    context = "\n\n".join(f"[DOC {i+1}] {doc}" for i, doc in enumerate(documents_rag[0]))
    contexts.append(context)


queue = deque()
for pair in zip(queries, contexts):
    queue.append(pair)

total_len = len(queue)
cooldown = 0.5

while len(queue) > 0:
    sleep(cooldown)
    print(f"Progress: {len(queue)}/{total_len}")
    try:
        query, context = queue.popleft()
        model_response = rag_query(query, context)
        responses.append((query,model_response))
    except Exception as e:
        queue.append((query, context))
        cooldown = min(60, cooldown * 2)
        print(e)

Progress: 6/6
Progress: 5/6
Progress: 4/6
Progress: 3/6
Progress: 2/6
Progress: 1/6


In [8]:
for question,model_response in responses:
    print("="*30,f"[Question]: {question}","="*30, sep="\n")
    display(Markdown(model_response))

[Question]: Bardzo boli mnie głowa i potrzebuje szybko zabić ból, mam uczulenie na salicyl, więc tak żebym się nie przekręcił


Analiza objawów: Użytkownik zgłasza silny ból głowy. Kluczową informacją jest uczulenie na salicylany.

Możliwe leki z kontekstu:

*   **Apap® migrena**
    *   **Działanie:** Lek zawiera paracetamol, kwas acetylosalicylowy i kofeinę. Paracetamol i kwas acetylosalicylowy zmniejszają ból i gorączkę. Kwas acetylosalicylowy dodatkowo działa przeciwzapalnie. Kofeina jest łagodnym środkiem pobudzającym i zwiększa działanie kwasu acetylosalicylowego i paracetamolu.
    *   **Wskazania:** Doraźne leczenie bólu głowy oraz napadów migreny (objawów takich jak: ból głowy, nudności, nadwrażliwość na światło i dźwięk oraz zaburzenia codziennego funkcjonowania) z aurą lub bez aury.
    *   **Przeciwwskazania (istotne w kontekście zapytania):** Nie stosować, jeżeli występuje uczulenie (nadwrażliwość) na kwas acetylosalicylowy, paracetamol, kofeinę lub którykolwiek ze składników tabletek powlekanych APAP migrena. Nie stosować, jeśli stwierdzono kiedykolwiek reakcje alergiczne na inne leki przeciwbólowe, przeciwzapalne lub przeciwgorączkowe, takie jak kwas acetylosalicylowy i salicylany.
    *   **Ocena w kontekście zapytania:** Lek zawiera kwas acetylosalicylowy (salicylan), na który użytkownik ma uczulenie. Jest przeciwwskazany.

*   **Coffepirine Tabletki od bólu głowy**
    *   **Działanie:** Lek zawiera kwas acetylosalicylowy o działaniu przeciwbólowym, przeciwgorączkowym i przeciwzapalnym oraz kofeinę, która działa na ośrodek naczynioruchowy w mózgu, wywołując skurcz mięśni gładkich naczyń krwionośnych, a tym samym zmniejszając przepływ krwi przez tkankę mózgową. Działa pobudzająco na ośrodek oddechowy.
    *   **Wskazania:** Stosuje się w leczeniu bólów różnego pochodzenia o małym umiarkowanym nasileniu (np. bóle głowy, reumatyczne, mięśni, nerwobóle, ból zęba).
    *   **Przeciwwskazania (istotne w kontekście zapytania):** Nie stosować, jeśli pacjent ma uczulenie na kwas acetylosalicylowy, kofeinę lub którykolwiek z pozostałych składników tego leku lub na salicylany.
    *   **Ocena w kontekście zapytania:** Lek zawiera kwas acetylosalicylowy i jest przeciwwskazany w przypadku uczulenia na salicylany.

*   **Cefalgin Migraplus**
    *   **Działanie:** Lek złożony zawierający paracetamol i propyfenazon, które wykazują działanie przeciwbólowe i przeciwgorączkowe, oraz niewielką ilość kofeiny. Kofeina nasila działanie przeciwbólowe paracetamolu.
    *   **Wskazania:** Bóle o słabym lub umiarkowanym nasileniu: migrenowe bóle głowy, bóle zębów, bóle miesiączkowe, nerwobóle; gorączka.
    *   **Przeciwwskazania (istotne w kontekście zapytania):** Nie stosować, jeśli pacjent ma uczulenie (nadwrażliwość) na paracetamol, kofeinę lub kwas acetylosalicylowy lub którykolwiek z pozostałych składników tego leku.
    *   **Ocena w kontekście zapytania:** Pomimo braku kwasu acetylosalicylowego w składzie, ulotka wymienia uczulenie na kwas acetylosalicylowy jako przeciwwskazanie. Ze względu na uczulenie użytkownika na salicylany, ten lek również jest przeciwwskazany.

Podsumowanie opcji:
Brak danych. Żaden z przedstawionych w kontekście leków nie jest odpowiedni do stosowania u osoby z uczuleniem na salicylany, ponieważ wszystkie wymienione w przeciwwskazaniach lub w składzie zawierają substancje (kwas acetylosalicylowy, salicylany) lub mają przeciwwskazania dotyczące kwasu acetylosalicylowego.

[Question]: Mam problemy ze snem, mam alergie na betalaktame


Analiza objawów:
Pacjent zgłasza problemy ze snem oraz alergię na betalaktamy. W kontekście alergii na betalaktamy należy ocenić skład leków, aby uniknąć potencjalnych reakcji alergicznych. Problemy ze snem są bezpośrednim wskazaniem do poszukiwania leku o działaniu nasennym.

Możliwe leki z kontekstu:

1.  **ApoDream**
    *   **Substancja czynna:** Zopiklon
    *   **Działanie:** Lek należy do grupy leków nasennych. Wpływa na pracę mózgu, ułatwiając sen.
    *   **Wskazania:** Leczenie problemów ze snem, takich jak trudności z zasypianiem, budzenie się w środku nocy, zbyt wczesne poranne budzenie się, poważne lub nieprzyjemne kłopoty ze snem wywołane przez nastrój lub problemy związane ze zdrowiem psychicznym. Może być stosowany w leczeniu przejściowych, jak i dłużej trwających problemów ze snem.
    *   **Przeciwwskazania:** Uczulenie na zopiklon lub którykolwiek z pozostałych składników tego leku, myasthenia gravis, ciężka niewydolność oddechowa, zespół bezdechu sennego, poważne problemy z wątrobą, wiek poniżej 18 lat. W składzie leku ApoDream nie wymieniono substancji należących do grupy betalaktamów.

2.  **AlergoTeva**
    *   **Substancja czynna:** Desloratadyna
    *   **Działanie:** Lek przeciwalergiczny o działaniu przeciwhistaminowym, który nie wywołuje senności.
    *   **Wskazania:** Łagodzenie objawów alergicznego zapalenia błony śluzowej nosa (np. kataru siennego, uczulenia na roztocza), takich jak kichanie, wodnista wydzielina lub swędzenie nosa, swędzenie podniebienia oraz swędzenie, zaczerwienienie lub łzawienie oczu. Stosowany jest również w celu łagodzenia objawów związanych z pokrzywką (świąd skóry i bąble pokrzywkowe). Złagodzenie tych objawów utrzymuje się przez cały dzień, co ułatwia powrót do normalnych codziennych czynności oraz normalnego snu.
    *   **Przeciwwskazania:** Uczulenie na desloratadynę lub którykolwiek z pozostałych składników tego leku, lub na loratadynę. W składzie leku AlergoTeva nie wymieniono substancji należących do grupy betalaktamów.

3.  **Beltavac® Polymerized**
    *   **Substancja czynna:** Ekstrakty alergenowe (Alternaria alternata)
    *   **Działanie:** Specyficzna i zindywidualizowana szczepionka terapeutyczna.
    *   **Wskazania:** Leczenie nadwrażliwości na choroby alergiczne IgE-zależne, takie jak nieżyt nosa, zapalenie spojówek i astma u dorosłych i dzieci w wieku od 5 lat, wywołane alergią na *Alternaria alternata*.
    *   **Przeciwwskazania:** Niekontrolowana lub ciężka astma, nowotwory złośliwe, aktywne układowe choroby autoimmunologiczne, nadwrażliwość na którąkolwiek substancję pomocniczą. W składzie leku Beltavac® Polymerized nie wymieniono substancji należących do grupy betalaktamów. Lek ten jest przeznaczony do leczenia alergii na konkretny alergen (Alternaria alternata), a nie na alergię na betalaktamy ani problemy ze snem.

Podsumowanie opcji:
Dla problemów ze snem bezpośrednio wskazany jest lek **ApoDream**, którego substancją czynną jest zopiklon. Zgodnie z ulotką, składniki leku ApoDream nie zawierają betalaktamów.
Lek **AlergoTeva** jest lekiem przeciwalergicznym, który może pośrednio ułatwiać normalny sen poprzez łagodzenie objawów alergii, jednak sam w sobie nie jest lekiem nasennym i nie wywołuje senności. Jego skład również nie zawiera betalaktamów.
Lek **Beltavac® Polymerized** jest szczepionką alergenową przeznaczoną do leczenia specyficznej alergii na Alternaria alternata i nie jest wskazany w leczeniu problemów ze snem ani alergii na betalaktamy.

[Question]: Mam ból nogi, bo się uderzyłem potrzebuje załagodzić bol


...

Analiza objawów:
Bóle żołądka mogą mieć różne przyczyny. W dostarczonym kontekście dostępne są leki, które mogą być stosowane w przypadku dolegliwości żołądkowo-jelitowych, w tym skurczowych bólów. Ważne jest zwrócenie uwagi na naturę bólu (np. skurczowy) oraz na wszelkie inne objawy towarzyszące, takie jak gorączka, nudności, wymioty, czy zmiany w rytmie wypróżnień, ponieważ mogą one wskazywać na potrzebę konsultacji lekarskiej.

Możliwe leki z kontekstu:

1.  **Boldaloin®**
    *   **Działanie:** Tradycyjnie stosowany jako środek pobudzający wydzielanie żółci i soku żołądkowego oraz regulujący częstość wypróżnień.
    *   **Wskazania:** Tradycyjnie w niestrawności z uczuciem pełności w jamie brzusznej, zaburzeniach wydzielania żółci i soku żołądkowego, w lekkich skurczowych zaburzeniach żołądkowo – jelitowych. Tradycyjnie w regulacji częstości wypróżnień (w łagodnych zaparciach związanych ze zmianą diety, miejsca pobytu). Skuteczność leku opiera się wyłącznie na długim okresie stosowania i doświadczeniu.
    *   **Przeciwwskazania:** Uczulenie na substancje czynne lub którykolwiek ze składników leku, marskość wątroby, niedrożność dróg żółciowych i przewodu pokarmowego, zwężenie przewodu pokarmowego, atonia jelit, zapalenie wyrostka robaczkowego, choroby zapalne jelita grubego (np. choroba Crohna, wrzodziejące zapalenie okrężnicy), bóle brzucha z nieznanych przyczyn, niewydolność nerek, biegunka, zaburzenia równowagi wodno-elektrolitowej. Nie stosować u dzieci w wieku poniżej 12 lat. Nie zaleca się stosowania w ciąży i podczas karmienia piersią.

2.  **AuroGastro**
    *   **Działanie:** Zawiera hioscyny butylobromek, który należy do grupy leków przeciwskurczowych.
    *   **Wskazania:** Stosowany w celu złagodzenia skurczów mięśni żołądka i jelit.
    *   **Przeciwwskazania:** Uczulenie na hioscyny butylobromek lub którykolwiek ze składników leku, rozrost gruczołu krokowego, zatrzymanie moczu, przeszkoda mechaniczna przewodu pokarmowego (zwężenie przewodu pokarmowego) lub zwężenie odźwiernika, porażenna lub obturacyjna niedrożność jelit, tachykardia, jaskra, choroba myasthenia gravis, rozdęcie okrężnicy (megacolon), rzadka choroba dziedziczna uniemożliwiająca przyjmowanie niektórych substancji pomocniczych.
    *   **Ostrzeżenia/Środki ostrożności:** Należy natychmiast skontaktować się z lekarzem lub farmaceutą, jeśli u pacjenta wystąpi niewyjaśniony ból brzucha, który utrzymuje się lub nasila, lub występuje razem z gorączką, złym samopoczuciem, uczuciem bycia chorym, zmianami rytmu wypróżnień, tkliwością brzucha, niskim ciśnieniem krwi, uczuciem omdlenia lub krwią w kale. Nie zaleca się stosowania w ciąży. Nie badano przenikania do mleka podczas karmienia piersią; leki tego typu mogą hamować produkcję mleka. Nie zaleca się stosowania u dzieci w wieku poniżej 6 lat.

3.  **Apo-Napro Fast**
    *   **Działanie:** Zawiera naproksen, który jest lekiem przeciwzapalnym, przeciwbólowym i przeciwgorączkowym (należy do grupy NLPZ).
    *   **Wskazania:** Leczenie dolegliwości bólowych o małym i umiarkowanym nasileniu, takich jak ból głowy, ból zęba, ból mięśni, ból stawów, ból pleców, bolesne miesiączkowanie, dolegliwości bólowe o niewielkim nasileniu związane z przeziębieniem. Obniżenie gorączki.
    *   **Przeciwwskazania:** Uczulenie na naproksen lub naproksen sodowy lub którykolwiek ze składników leku, reakcja alergiczna (astma, katar, świąd) po zastosowaniu aspiryny, ibuprofenu lub innych NLPZ, krwawienie z przewodu pokarmowego lub perforacja po NLPZ w przeszłości, nawracające wrzody żołądka/dwunastnicy lub krwawienia, choroba wrzodowa żołądka lub dwunastnicy, zapalenie błony śluzowej żołądka lub ból żołądka, krwawienia wewnętrzne, tendencje do krwawień lub leczenie lekami przeciwzakrzepowymi, ciężkie zaburzenia czynności nerek, ciężkie zaburzenia czynności wątroby, ciężka niewydolność serca, ostatnie trzy miesiące ciąży.
    *   **Ważna uwaga dotycząca bólu żołądka:** Lek ten nie jest odpowiedni do leczenia bólu spowodowanego dolegliwościami żołądkowo-jelitowymi.

Podsumowanie opcji:
Dla dolegliwości związanych z bólami żołądka, szczególnie o charakterze skurczowym, najbardziej adekwatne wydają się być leki **AuroGastro** (bezpośrednio wskazany na skurcze żołądka i jelit) oraz **Boldaloin®** (wskazany na lekkie skurczowe zaburzenia żołądkowo-jelitowe oraz niestrawność). W przypadku stosowania obu tych leków należy zwrócić szczególną uwagę na przeciwwskazania oraz ostrzeżenia, zwłaszcza dotyczące bólu brzucha z nieznanych przyczyn lub bólu utrzymującego się/nasilającego się, lub występującego z innymi niepokojącymi objawami, co wymaga konsultacji lekarskiej. Lek **Apo-Napro Fast** nie jest przeznaczony do leczenia bólów żołądka i posiada wiele przeciwwskazań związanych z przewodem pokarmowym, w tym sam ból żołądka.

[Question]: Boli mnie gardełko, pomóż szybko


Na podstawie zgłoszonego objawu "ból gardła" dostępne są preparaty mające na celu jego łagodzenie.

**Możliwe leki z kontekstu:**

1.  **Cevitt Gardło**
    *   **Działanie:** Preparat zawiera hialuronian sodu, ksantan, karbomer, witaminę C oraz cynk. Składniki te tworzą na błonie śluzowej ochronny film, który nawilża ją i łagodzi podrażnienia.
    *   **Wskazania:** Stosowanie w leczeniu podrażnień wywołujących kaszel, podrażnień błony śluzowej jamy ustnej i gardła oraz związanym z nimi bólem gardła, utrudnionym przełykaniem oraz chrypką. Preparat jest również polecany osobom palącym papierosy, osobom ze zgagą oraz pracującym głosem.
    *   **Przeciwwskazania:** brak danych.

2.  **Benzydamine neo-angin forte**
    *   **Substancja czynna:** Chlorowodorek benzydaminy, należący do niesteroidowych leków przeciwzapalnych (NLPZ).
    *   **Działanie:** Działa przeciwbólowo i przeciwobrzękowo (przeciwzapalnie).
    *   **Wskazania:** Objawowe miejscowe leczenie ostrego bólu gardła z towarzyszącymi typowymi objawami stanu zapalnego, takimi jak ból, zaczerwienienie lub obrzęk w jamie ustnej i gardle. Preparat jest przeznaczony do stosowania u dorosłych.
    *   **Przeciwwskazania:** Uczulenie na chlorowodorek benzydaminy lub którykolwiek z pozostałych składników leku. Uczulenie na salicylany (np. kwas acetylosalicylowy, kwas salicylowy) lub inne leki z grupy NLPZ. Astma oskrzelowa lub choroby alergiczne, ze względu na zwiększone ryzyko wystąpienia skurczu oskrzeli lub alergii. Nie należy stosować w ciąży, chyba że jest to bezwzględnie konieczne i zalecone przez lekarza. Leku nie powinno się stosować w okresie karmienia piersią.
    *   **Dawkowanie:** Stosować na ogół od 2 do 6 razy na dobę (nie częściej niż co 1,5 – 3 godziny), od 2 do 4 dawek na podanie. Leczenie ciągłe nie powinno trwać dłużej niż 7 dni. Podczas rozpylania leku należy wstrzymać oddech. Nie należy stosować bezpośrednio przed jedzeniem lub piciem.

3.  **Benzydamine neo-angin**
    *   **Substancja czynna:** Chlorowodorek benzydaminy, należący do niesteroidowych leków przeciwzapalnych (NLPZ).
    *   **Działanie:** Działa przeciwbólowo i przeciwobrzękowo (przeciwzapalnie).
    *   **Wskazania:** Objawowe miejscowe leczenie ostrego bólu gardła z towarzyszącymi typowymi objawami stanu zapalnego, takimi jak ból, zaczerwienienie lub obrzęk w jamie ustnej i gardle. Preparat jest wskazany do stosowania u dorosłych i dzieci w wieku powyżej 6 lat.
    *   **Przeciwwskazania:** Uczulenie na chlorowodorek benzydaminy lub którykolwiek z pozostałych składników leku. Uczulenie na salicylany lub inne leki z grupy NLPZ. Astma oskrzelowa lub choroby alergiczne, ze względu na zwiększone ryzyko wystąpienia skurczu oskrzeli lub alergii. Nie należy stosować w ciąży, chyba że jest to bezwzględnie konieczne i zalecone przez lekarza. Leku nie powinno się stosować w okresie karmienia piersią. Nie zaleca się stosowania u dzieci, które nie potrafią wstrzymać oddechu podczas stosowania aerozolu.
    *   **Dawkowanie:** U dorosłych i dzieci powyżej 12 lat należy rozpylać od 4 do 8 dawek na podanie. U dzieci w wieku od 6 do 12 lat należy rozpylać 4 dawki na podanie. Lek stosuje się na ogół od 2 do 6 razy na dobę (nie częściej niż co 1,5 – 3 godziny). Leczenie ciągłe nie powinno trwać dłużej niż 7 dni. Podczas rozpylania leku należy wstrzymać oddech. Nie należy stosować bezpośrednio przed jedzeniem lub piciem.

**Podsumowanie opcji:**
Dla objawu "ból gardła" dostępne są tabletki do ssania Cevitt Gardło, które łagodzą podrażnienia, oraz aerozole Benzydamine neo-angin forte i Benzydamine neo-angin, które wykazują działanie przeciwbólowe i przeciwzapalne. Aerozole z benzydaminą są szczególnie wskazane w przypadku ostrego bólu gardła z towarzyszącymi objawami stanu zapalnego (zaczerwienienie, obrzęk). Benzydamine neo-angin forte jest przeznaczony dla dorosłych, natomiast Benzydamine neo-angin może być stosowany przez dorosłych i dzieci powyżej 6 lat. Przed zastosowaniem któregokolwiek z wymienionych preparatów należy zapoznać się ze wszystkimi informacjami zawartymi w ulotce, w tym z przeciwwskazaniami i zaleceniami dotyczącymi dawkowania.

[Question]: Mam zatkane zatoki


Analiza objawów:
Stwierdzono objaw zatkanych zatok.

Możliwe leki z dostarczonego kontekstu:

1.  **Acatar Zatoki**
    *   **Działanie:** Acatar Zatoki zawiera ibuprofen, który działa przeciwbólowo, przeciwzapalnie i przeciwgorączkowo, oraz pseudoefedryny chlorowodorek, który zmniejsza obrzęk błony śluzowej nosa.
    *   **Wskazania:** Stosowany doraźnie w celu złagodzenia objawów grypy i przeziębienia, takich jak ból i niedrożność zatok obocznych nosa, katar, ból głowy, gorączka, bóle stawowo-mięśniowe.
    *   **Dawkowanie (dorośli i dzieci powyżej 12 lat):** 1 do 2 tabletek doustnie co 4 godziny po posiłkach. Nie należy stosować dawki większej niż 6 tabletek na dobę. Nie wolno przyjmować leku przez okres ponad 3 dni bez konsultacji z lekarzem.
    *   **Przeciwwskazania (wybrane):** Uczulenie na substancje czynne lub inne NLPZ, astma oskrzelowa po przyjęciu kwasu acetylosalicylowego lub innych NLPZ, choroba wrzodowa żołądka i (lub) dwunastnicy (czynna lub przebyta), ciężka niewydolność wątroby, serca lub nerek, jednoczesne przyjmowanie innych NLPZ, ciąża i okres karmienia piersią, skaza krwotoczna, ciężkie zaburzenia układu sercowo-naczyniowego, tachykardia, dławica piersiowa, ciężkie lub niekontrolowane nadciśnienie tętnicze, nadczynność tarczycy, cukrzyca, jaskra z zamkniętym kątem, rozrost gruczołu krokowego, guz chromochłonny nadnerczy.

2.  **Aspirin® Complex Zatoki**
    *   **Działanie:** Lek zawiera kwas acetylosalicylowy, który ma właściwości przeciwbólowe, przeciwzapalne i przeciwgorączkowe, oraz chlorowodorek pseudoefedryny, który zmniejsza obrzęk i przekrwienie błony śluzowej nosa.
    *   **Wskazania:** Stosowany w leczeniu objawowym obrzęku błony śluzowej nosa i (lub) zatok (zapalenie błony śluzowej nosa, zapalenie zatok) oraz bólu i gorączki związanych z przeziębieniem i (lub) objawami grypopodobnymi. Przeznaczony dla dorosłych i młodzieży w wieku od 16 lat.
    *   **Dawkowanie (dorośli i młodzież od 16 roku życia):** Dawka pojedyncza to 1-2 saszetki, którą można powtórzyć po upływie 4 godzin. Maksymalna dawka dobowa to 6 saszetek. Nie należy zażywać tego leku dłużej niż 3 dni bez konsultacji z lekarzem.
    *   **Przeciwwskazania (wybrane):** Uczulenie na kwas acetylosalicylowy, chlorowodorek pseudoefedryny lub inne salicylany/NLPZ, astma spowodowana podaniem salicylanów/NLPZ, choroba wrzodowa żołądka, zwiększona skłonność do krwawień, niewydolność wątroby, ciężka ostra lub przewlekła choroba nerek/niewydolność nerek, ciężka niewydolność serca, ciężka choroba wieńcowa, bardzo wysokie lub niekontrolowane nadciśnienie tętnicze, przyjmowanie metotreksatu w dawce 15 mg na tydzień lub większej, ciąża i karmienie piersią, przyjmowanie inhibitorów MAO (lub w ciągu ostatnich dwóch tygodni), jaskra z zamkniętym kątem przesączania, zatrzymanie moczu.

3.  **Afrin ND**
    *   **Działanie:** Afrin ND zawiera oksymetazoliny chlorowodorek, który jest lekiem udrożniającym górne drogi oddechowe. Działa poprzez zwężenie naczyń krwionośnych w nosie, przynosząc ulgę przy zatkanym nosie. Zaczyna działać w ciągu kilku minut.
    *   **Wskazania:** Stosowany w celu leczenia objawów zatkanego nosa, spowodowanych katarem siennym, przeziębieniem i zapaleniem zatok.
    *   **Dawkowanie:** Dorośli i dzieci w wieku > 10 lat: 1-2 dawki aerozolu do każdego otworu nosowego, co 12 godzin. Dzieci w wieku 6-10 lat: 1 dawka aerozolu do każdego otworu nosowego, co 12 godzin. Nie należy stosować więcej niż 8 dawek aerozolu (u dorosłych) lub 4 dawek (u dzieci) w ciągu 24 godzin. Nie należy stosować leku dłużej niż 7 dni.
    *   **Przeciwwskazania (wybrane):** Uczulenie na chlorowodorek oksymetazoliny lub którykolwiek ze składników leku, leczenie inhibitorami monoaminooksydazy (MAO), jaskra z wąskim kątem przesączania, po przezklinowym usunięciu przysadki, suche zanikowe zapalenie błony śluzowej nosa, ostra choroba wieńcowa i lewokomorowa niewydolność serca. Nie stosować u dzieci poniżej 6 lat.

Podsumowanie opcji:
Acatar Zatoki i Aspirin® Complex Zatoki to leki doustne o złożonym działaniu, które łączą składnik przeciwbólowy/przeciwzapalny/przeciwgorączkowy (ibuprofen lub kwas acetylosalicylowy) z substancją udrażniającą nos (pseudoefedryna). Mogą być adekwatne, jeśli zatkanym zatokom towarzyszy ból lub gorączka. Afrin ND to aerozol do nosa zawierający oksymetazolinę, działającą miejscowo na udrożnienie nosa i zatok. Jest to opcja skupiona wyłącznie na objawach zatkanego nosa i zatok, z szybkim początkiem działania. Wybór zależy od preferowanej formy podania i zakresu objawów.